## CASE — ¿la RR estimada desde la PPG puede reemplazar a la banda torácica para detectar activación?
**Pregunta:** al clasificar activación emocional (arousal) con señales de un solo sensor de PPG,
¿añadir la RR estimada desde la PPG aporta lo mismo que añadir la RR de la banda torácica?

**Diseño (fijado antes de ver resultados):**
- Ventanas de 30 s con paso de 10 s, solo dentro de un vídeo emocional, sin artefacto en la PPG y con
  RR de banda y de PPG disponibles. **El mismo conjunto de ventanas para todos los brazos.**
- **Etiqueta principal (diseño de CASE):** activación alta = vídeos divertidos y de miedo;
  baja = aburridos y relajantes. **Secundaria:** arousal autoinformado (joystick) por encima de la
  mediana del propio participante, si están los archivos de anotaciones.
- Características (cada una normalizada dentro del participante, sin usar etiquetas):
  HRV de la PPG (FC media, RMSSD, SDNN); RR de la banda (conteo); RR de la PPG con la variante elegida
  en el notebook 09 (prueba en CASE: MAE 1.62 rpm) y con la original (RIIV ≥6).
- Regresión logística, validación dejando un participante fuera (30 pliegues).
  Métrica: AUC por participante. Comparaciones pareadas por participante (Wilcoxon).

In [3]:
# e-1 — Imports, configuración y rutas
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, decimate, detrend, filtfilt, find_peaks, resample_poly
from scipy.stats import wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = BASE_DIR / "results"
CFG = json.loads((OUT / "config_congelada.json").read_text())
VAR = json.loads((OUT / "config_case_correccion.json").read_text())["variante"]   # p. ej. "Mediana 3 ≥10"
MOD_ELEGIDA, LO_ELEGIDO = VAR.rsplit(" ≥", 1)[0], float(VAR.rsplit("≥", 1)[1])
FS = 125
VENTANA_S, PASO_S = CFG["ventana_s"], CFG["paso_s"]
RPM_MAX = CFG["rpm_max"]
N = VENTANA_S * FS
CASE_DIR = BASE_DIR / "data" / "CASE" / "data" / "interpolated"
ARCHIVOS = sorted((CASE_DIR / "physiological").glob("sub_*.csv"), key=lambda p: int(p.stem.split("_")[1]))
ACTIVACION = {1: 1, 2: 1, 7: 1, 8: 1, 3: 0, 4: 0, 5: 0, 6: 0}      # divertido/miedo = alta; aburrido/relajante = baja
CATEGORIA = {1: "divertido", 2: "divertido", 3: "aburrido", 4: "aburrido",
             5: "relajante", 6: "relajante", 7: "miedo", 8: "miedo"}
print(len(ARCHIVOS), "participantes | variante de RR desde PPG:", VAR)

30 participantes | variante de RR desde PPG: Mediana 3 ≥10


In [4]:
# e-2 — Funciones (las mismas del notebook 09, más HRV)
def pico(x, fs, nfft, lo):
    x = detrend(x) * np.hanning(len(x))
    f = np.fft.rfftfreq(nfft, 1 / fs) * 60
    P = np.abs(np.fft.rfft(x, nfft)) ** 2
    b = (f >= lo) & (f <= RPM_MAX)
    return f[np.argmax(np.where(b, P, -np.inf))]

def series_latidos(ppg):
    b, a = butter(2, [0.5, 8], btype="band", fs=FS)
    x = filtfilt(b, a, ppg)
    esc = 1.4826 * np.median(np.abs(x - np.median(x)))
    pk, _ = find_peaks(x, distance=int(0.33 * FS), prominence=0.5 * esc)
    vl = np.array([p0 + np.argmin(x[p0:p1]) for p0, p1 in zip(pk[:-1], pk[1:])], int)
    pk = pk[1:]
    t = pk / FS
    ibi = np.diff(t)
    t, pk, vl = t[1:], pk[1:], vl[1:]
    med = pd.Series(ibi).rolling(11, center=True, min_periods=3).median().to_numpy()
    ok = (ibi >= 0.33) & (ibi <= 1.5) & (np.abs(ibi - med) <= 0.3 * med)
    y = ppg - np.median(ppg)
    return t[ok], (y[pk] - y[vl])[ok], ibi[ok]

def rr_serie(t, v, t0, lo):
    m = (t >= t0) & (t < t0 + VENTANA_S)
    if m.sum() < 15:
        return np.nan
    tg = t0 + np.arange(VENTANA_S * 4) / 4
    return pico(np.interp(tg, t[m], v[m]), 4, 1024, lo)

def hrv(t, ibi, t0):
    """FC media, RMSSD y SDNN de los latidos limpios de la ventana.
    RMSSD solo con pares de latidos consecutivos (sin un latido descartado entre ellos)."""
    m = (t >= t0) & (t < t0 + VENTANA_S)
    if m.sum() < 15:
        return np.nan, np.nan, np.nan
    tw, iw = t[m], ibi[m]
    consec = np.abs(np.diff(tw) - iw[1:]) < 0.02
    d = np.diff(iw)[consec]
    rmssd = 1000 * np.sqrt(np.mean(d ** 2)) if consec.sum() >= 10 else np.nan
    return 60 / iw.mean(), rmssd, 1000 * iw.std()

def marcas_banda(rsp, fs=1000):
    x = decimate(decimate(rsp, 10, zero_phase=True), 4, zero_phase=True)
    f = fs / 40
    b, a = butter(2, [0.05, 1.0], btype="band", fs=f)
    x = filtfilt(b, a, x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    pk, _ = find_peaks(x, distance=int(f), prominence=0.25 * iqr)
    return pk / f

def rr_ref(m, t0, t1, min_marcas=3):
    m = m[(m >= t0) & (m < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

def rr_ppg(x5, t, iav, ifv, t0, mod, lo):
    e = {"RIIV": pico(x5, 5, 2048, lo), "RIAV": rr_serie(t, iav, t0, lo), "RIFV": rr_serie(t, ifv, t0, lo)}
    if mod != "Mediana 3":
        return e[mod]
    v = np.array(list(e.values()))
    return np.nanmedian(v) if np.isfinite(v).sum() >= 2 else np.nan

In [5]:
# e-3 — Ventanas y características
filas = []
for p in ARCHIVOS:
    d = pd.read_csv(p, usecols=["daqtime", "bvp", "rsp", "video"])
    vid = d["video"].to_numpy(int)
    ppg = resample_poly(d["bvp"].to_numpy(float), FS, 1000)
    marcas = marcas_banda(d["rsp"].to_numpy(float))
    b, a = butter(4, CFG["lp_fft_hz"], btype="low", fs=FS)
    riiv = filtfilt(b, a, ppg)
    b, a = butter(4, CFG["lp_art_hz"], btype="low", fs=FS)
    riiv_art = filtfilt(b, a, ppg)
    n5 = 5 * FS
    tipico = np.median([np.std(riiv_art[i:i + n5]) for i in range(0, len(ppg) - n5 + 1, n5)])
    t, iav, ifv = series_latidos(ppg)
    # Anotaciones continuas (opcional): arousal medio del joystick en la ventana
    fa = CASE_DIR / "annotations" / p.name
    ann = pd.read_csv(fa) if fa.exists() else None
    if ann is not None and not {"jstime", "arousal"} <= set(ann.columns):
        ann = None
    t_daq0 = d["daqtime"].iloc[0]
    for t0 in np.arange(0, len(ppg) / FS - VENTANA_S + 1e-9, PASO_S):
        v = vid[int(t0 * 1000):int((t0 + VENTANA_S) * 1000)]
        if v.min() != v.max() or v[0] not in ACTIVACION:
            continue
        i0 = int(round(t0 * FS))
        exc = max(np.std(riiv_art[i:i + n5]) for i in range(i0, i0 + N - n5 + 1, n5)) / tipico
        x5 = decimate(decimate(riiv[i0:i0 + N], 5, zero_phase=True), 5, zero_phase=True)
        fc, rmssd, sdnn = hrv(t, ifv, t0)
        arousal = np.nan
        if ann is not None:
            ta = t_daq0 + 1000 * t0
            arousal = ann.loc[(ann["jstime"] >= ta) & (ann["jstime"] < ta + 1000 * VENTANA_S), "arousal"].mean()
        filas.append({"Participante": p.stem, "Inicio (s)": t0, "Vídeo": int(v[0]),
                      "Categoría": CATEGORIA[int(v[0])], "Activación (vídeo)": ACTIVACION[int(v[0])],
                      "Arousal joystick": arousal, "Excursión": exc,
                      "FC": fc, "RMSSD": rmssd, "SDNN": sdnn,
                      "RR banda": rr_ref(marcas, t0, t0 + VENTANA_S),
                      "RR PPG elegida": rr_ppg(x5, t, iav, ifv, t0, MOD_ELEGIDA, LO_ELEGIDO),
                      "RR PPG original": pico(x5, 5, 2048, 6)})
E = pd.DataFrame(filas)
E["Válida"] = ((E["Excursión"] <= CFG["umbral_art"])
               & E[["FC", "RMSSD", "SDNN", "RR banda", "RR PPG elegida", "RR PPG original"]].notna().all(axis=1)
               & E["RR banda"].between(6, RPM_MAX))
E.to_csv(OUT / "case_estudio_ventanas.csv", index=False)
print(f"{len(E)} ventanas | válidas {E['Válida'].sum()} ({100 * E['Válida'].mean():.0f} %)")
print(E[E["Válida"]].groupby("Categoría").size().to_dict())

3090 ventanas | válidas 2750 (89 %)
{'aburrido': 591, 'divertido': 833, 'miedo': 708, 'relajante': 618}


In [6]:
# e-4 — Normalización dentro del participante y etiquetas
CARAC = ["FC", "RMSSD", "SDNN", "RR banda", "RR PPG elegida", "RR PPG original"]
V = E[E["Válida"]].copy()
V["RMSSD"] = np.log(V["RMSSD"])           # distribución sesgada
V["SDNN"] = np.log(V["SDNN"])
Z = V.groupby("Participante")[CARAC].transform(lambda c: (c - c.mean()) / (c.std() + 1e-9))
ETIQUETAS = {"Vídeo (diseño)": V["Activación (vídeo)"]}
if V["Arousal joystick"].notna().mean() > 0.9:
    med = V.groupby("Participante")["Arousal joystick"].transform("median")
    ETIQUETAS["Joystick (> mediana propia)"] = (V["Arousal joystick"] > med).astype(int)
else:
    print("Sin anotaciones de joystick: solo se usa la etiqueta del vídeo.")

# Correlación de las RR con la banda en este mismo conjunto de ventanas
for col in ["RR PPG elegida", "RR PPG original"]:
    e = (V[col] - V["RR banda"]).abs()
    print(f"{col}: MAE mediano por participante {e.groupby(V['Participante']).mean().median():.2f} rpm | "
          f"r dentro de participante {V.groupby('Participante').apply(lambda g: g[col].corr(g['RR banda'])).median():.2f}")

RR PPG elegida: MAE mediano por participante 2.33 rpm | r dentro de participante 0.19
RR PPG original: MAE mediano por participante 10.41 rpm | r dentro de participante -0.01


In [7]:
# e-5 — Clasificación dejando un participante fuera
BRAZOS = {
    "HRV (PPG)":                     ["FC", "RMSSD", "SDNN"],
    "HRV + RR banda":                ["FC", "RMSSD", "SDNN", "RR banda"],
    "HRV + RR PPG (elegida)":        ["FC", "RMSSD", "SDNN", "RR PPG elegida"],
    "HRV + RR PPG (original)":       ["FC", "RMSSD", "SDNN", "RR PPG original"],
    "RR banda sola":                 ["RR banda"],
    "RR PPG sola (elegida)":         ["RR PPG elegida"],
}
PART = V["Participante"].to_numpy()
AUC = []
for et, y in ETIQUETAS.items():
    y = y.to_numpy()
    for brazo, cols in BRAZOS.items():
        X = Z[cols].to_numpy()
        for s in np.unique(PART):
            te = PART == s
            if len(np.unique(y[te])) < 2:
                continue
            clf = LogisticRegression(max_iter=1000).fit(X[~te], y[~te])
            AUC.append({"Etiqueta": et, "Brazo": brazo, "Participante": s,
                        "AUC": roc_auc_score(y[te], clf.predict_proba(X[te])[:, 1])})
A = pd.DataFrame(AUC)
A.to_csv(OUT / "case_estudio_auc.csv", index=False)

In [8]:
# e-6 — Resultados y comparaciones pareadas
for et, g in A.groupby("Etiqueta", sort=False):
    print(f"\n=== Etiqueta: {et} ===")
    T = g.groupby("Brazo", sort=False)["AUC"].agg(["count", "median", "mean", "std"]).round(3)
    print(T.rename(columns={"count": "Participantes", "median": "AUC mediana",
                            "mean": "AUC media", "std": "DE"}).to_string())
    W = g.pivot(index="Participante", columns="Brazo", values="AUC")
    for a, b in [("HRV + RR banda", "HRV (PPG)"), ("HRV + RR PPG (elegida)", "HRV (PPG)"),
                 ("HRV + RR PPG (elegida)", "HRV + RR banda"),
                 ("HRV + RR PPG (elegida)", "HRV + RR PPG (original)")]:
        dlt = (W[a] - W[b]).dropna()
        p = wilcoxon(dlt).pvalue if (dlt != 0).any() else 1.0
        print(f"{a}  vs  {b}: Δ AUC mediana {dlt.median():+.3f} (IC95 bootstrap "
              f"{np.percentile([np.median(np.random.default_rng(i).choice(dlt, len(dlt))) for i in range(2000)], [2.5, 97.5]).round(3)}), "
              f"Wilcoxon p = {p:.3f}")


=== Etiqueta: Vídeo (diseño) ===
                         Participantes  AUC mediana  AUC media     DE
Brazo                                                                
HRV (PPG)                           30        0.651      0.608  0.175
HRV + RR banda                      30        0.723      0.697  0.163
HRV + RR PPG (elegida)              30        0.640      0.605  0.172
HRV + RR PPG (original)             30        0.647      0.605  0.176
RR banda sola                       30        0.725      0.690  0.165
RR PPG sola (elegida)               30        0.528      0.483  0.108
HRV + RR banda  vs  HRV (PPG): Δ AUC mediana +0.125 (IC95 bootstrap [0.02  0.159]), Wilcoxon p = 0.003
HRV + RR PPG (elegida)  vs  HRV (PPG): Δ AUC mediana +0.003 (IC95 bootstrap [-0.009  0.01 ]), Wilcoxon p = 0.808
HRV + RR PPG (elegida)  vs  HRV + RR banda: Δ AUC mediana -0.127 (IC95 bootstrap [-0.162 -0.009]), Wilcoxon p = 0.001
HRV + RR PPG (elegida)  vs  HRV + RR PPG (original): Δ AUC mediana +0.00

In [9]:
# %% [e-7 — ¿Cuánto cambia la RR entre activación alta y baja, dentro de cada participante?]
g = V.groupby(["Participante", "Activación (vídeo)"])[["RR banda", "RR PPG elegida"]].mean().unstack()
dif = pd.DataFrame({c: g[(c, 1)] - g[(c, 0)] for c in ["RR banda", "RR PPG elegida"]})
print(dif.describe().round(2).to_string())
print("\nCorrelación entre participantes del cambio (banda vs PPG):",
      round(dif["RR banda"].corr(dif["RR PPG elegida"]), 2))

       RR banda  RR PPG elegida
count     30.00           30.00
mean       1.42           -0.04
std        1.29            1.10
min       -1.70           -2.64
25%        0.70           -0.60
50%        1.50            0.01
75%        2.35            0.65
max        4.27            2.10

Correlación entre participantes del cambio (banda vs PPG): 0.31
